### Importing necessary libraries

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import requests
from http.cookiejar import MozillaCookieJar
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
import gradio as gr

### trying llama model

In [12]:
# model = "llama3.2"
# base_url = "http://localhost:11434"

# llm = ChatOllama(
#     model=model,
#     base_url=base_url,
#     temperature=0.2
# )

##### The ChatOllama model are not that good. The model was unnecessarily hallucinating so i am switching to ChatGroq model with llama-3.3-70b-versatile which have over 70 billion parameters

### Trying groq model

In [13]:
load_dotenv()
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.2)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Loading Transcript

In [14]:
def get_vid_id(url: str):
    if url[0:4] == "http":
        start = url.find("=") + 1
        end = url.find("&")
        if end == -1:
            vid_id = url[start:]
        else:
            vid_id = url[start:end]
    else:
        vid_id = url
    if len(vid_id) == 11:
        return vid_id
    else:
        return None

In [15]:
def get_transcript(vid_id: str, cookie_file: str = "youtube_cookies.txt") -> str:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    })
    cookie_jar = MozillaCookieJar(cookie_file)
    cookie_jar.load(ignore_discard=True, ignore_expires=True)
    session.cookies = cookie_jar

    yt_api = YouTubeTranscriptApi(http_client=session)
    transcript_list = yt_api.fetch(vid_id, languages=["en"])
    return " ".join(chunk.text for chunk in transcript_list)

In [16]:
def build_retriever(transcript: str):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.create_documents([transcript])
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    return vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [17]:


parser = StrOutputParser()

In [18]:

prompt = PromptTemplate(
        template="""
    You are a helpful assistant that answers questions strictly based on the YouTube video transcript provided.
    If the answer is not found in the transcript, respond with: "I don't know — this topic isn't covered in the video."
    Do not make up or infer anything beyond what is in the transcript.

    Transcript Context:
    {context}

    Question: {question}

    Answer:""",
        input_variables=['context', 'question']
    )

In [21]:
import gradio as gr

retriever_store = {"vec": None}


def process_video(url):
    vid_id = get_vid_id(url)

    if vid_id:
        transcript = get_transcript(vid_id)
        if transcript:
            vec = build_retriever(transcript)
            retriever_store["vec"] = vec
            return "🎉 Transcript loaded! You can now ask questions.", gr.update(visible=True)
        else:
            return "🥹 Transcript not found! 🫷", gr.update(visible=False)
    else: 
        return "🥹 Vid_id not found! 🫷", gr.update(visible=False)



def answer_question(question):
    vec = retriever_store.get("vec")

    if vec is None:
        return "⚠️ Please load a video first."

    qa_chain = (
        {
            "context": RunnableLambda(lambda x: vec.invoke(question)) | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | parser
    )

    return qa_chain.invoke(question)


def reset_app():
    retriever_store["vec"] = None
    return "", "", "", "App reset.", gr.update(visible=False)


with gr.Blocks() as app:
    gr.Markdown("## ChatMyVideo_V2  😎")

    url_input = gr.Textbox(label="🫠Enter YouTube URL / Video ID")
    
    with gr.Row():
        load_btn = gr.Button("🔃 Load Video")
        reset_btn = gr.Button("🔙 Reset")

    status_output = gr.Textbox(label="Status 🕵️‍♀️")

    with gr.Column(visible=False) as qa_section:
        question_input = gr.Textbox(label="🙋 Ask a Question")
        ask_btn = gr.Button("Ask")
        answer_output = gr.Textbox(label="🤯  Answer")

    load_btn.click(
        process_video,
        inputs=url_input,
        outputs=[status_output, qa_section]
    )

    ask_btn.click(
        answer_question,
        inputs=question_input,
        outputs=answer_output
    )

    reset_btn.click(
        reset_app,
        inputs=[],
        outputs=[url_input, question_input, answer_output, status_output, qa_section]
    )

app.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [ ]:
## jPpsYzlowWU


